In [1]:
import re
import pandas as pd

# 1. Load and combine both Kaggle CSVs
df_syn = pd.read_csv("../data/raw/grievances_synthetic.csv", encoding="utf-8")
df_hold = pd.read_csv("../data/raw/grievances_holdout_templates.csv", encoding="utf-8")
df = pd.concat([df_syn, df_hold], ignore_index=True).drop_duplicates(subset=["text"]).copy()

# 2. Smart language classifier (Hindi / Hinglish / English)
HINGLISH_MARKERS = {
    "hai", "hain", "mein", "me", "nahi", "nhi", "kripya", "krpya", "kya",
    "ho", "gayi", "gaya", "raha", "rahi", "bahut", "se", "ke", "ki", "ka",
    "ko", "par", "pe", "karo", "kar", "paani", "pani", "sadak", "kachra", "nali", "naali"
}

def classify_language(text):
    text_str = str(text)
    dev_chars = len(re.findall(r"[\u0900-\u097F]", text_str))
    lat_chars = len(re.findall(r"[A-Za-z]", text_str))
    total_chars = dev_chars + lat_chars
    
    if total_chars == 0:
        return "Unknown"
    # If at least 30% of letters are Devanagari, label as Hindi
    if (dev_chars / total_chars) >= 0.30:
        return "Hindi"
    # Otherwise check for Hinglish markers in Roman script
    words = set(re.findall(r"[a-z]+", text_str.lower()))
    if len(words.intersection(HINGLISH_MARKERS)) >= 2:
        return "Hinglish"
    return "English"

# 3. Map Kaggle categories + keyword rules to your exact 8 categories
def map_category(row):
    text = str(row["text"])
    cat = str(row["category"])
    
    # Priority 1: Keyword rules for the 3 hidden categories
    if re.search(r"drain|sewer|sewage|naali|nali|waterlog|नाली|सीवर|जलभराव", text, re.I):
        return "Drainage & Sewage"
    if re.search(r"mosquito|fogging|stray|dengue|malaria|hygiene|मच्छर|फॉगिंग|डेंगू|मक्खी", text, re.I):
        return "Public Health & Hygiene"
    if re.search(r"street\s*light|streetlight|lamp|स्ट्रीट\s*लाइट|खंभा", text, re.I):
        return "Streetlights"
    
    # Priority 2: Direct category mappings
    direct_map = {
        "Sanitation & Garbage": "Waste & Sanitation",
        "Roads & Infrastructure": "Roads & Potholes",
        "Water Supply": "Water Supply",
        "Municipal Certificates": "Municipal Certificates"
    }
    return direct_map.get(cat, None)

df["mapped_category"] = df.apply(map_category, axis=1)
df["language"] = df["text"].apply(classify_language)

# 4. Build the In-Scope DataFrame in your exact 6-column schema
in_scope = df[df["mapped_category"].notna()].copy()
in_scope_clean = pd.DataFrame({
    "complaint_id": [f"KAG_{i:04d}" for i in range(1, len(in_scope) + 1)],
    "complaint_text": in_scope["text"].values,
    "category": in_scope["mapped_category"].values,
    "language": in_scope["language"].values,
    "source_type": "synthetic_generated",
    "source_reference": "Kaggle: abhisheksingh016/citizen-grievance-dataset"
})

# 5. Build a 60-row Out-of-Scope sample (Police, Banking, Education, etc.)
out_cats = ["Police & Law and Order", "Banking & Financial Services", "Education & Schools", "Employment & Labour"]
out_scope = df[df["category"].isin(out_cats)].sample(n=60, random_state=42)
out_scope_clean = pd.DataFrame({
    "complaint_id": [f"OOS_{i:04d}" for i in range(1, len(out_scope) + 1)],
    "complaint_text": out_scope["text"].values,
    "category": "Out of Scope",
    "language": out_scope["language"].values,
    "source_type": "synthetic_generated",
    "source_reference": "Kaggle: abhisheksingh016/citizen-grievance-dataset"
})

# 6. Save both files to data/processed/
in_scope_clean.to_csv("../data/processed/kaggle_in_scope.csv", index=False, encoding="utf-8")
out_scope_clean.to_csv("../data/processed/kaggle_out_of_scope.csv", index=False, encoding="utf-8")

print(f"Saved {len(in_scope_clean)} in-scope rows to data/processed/kaggle_in_scope.csv")
print(f"Saved {len(out_scope_clean)} out-of-scope rows to data/processed/kaggle_out_of_scope.csv\n")

print("=== CATEGORY x LANGUAGE COVERAGE FROM KAGGLE ===")
print(pd.crosstab(in_scope_clean["category"], in_scope_clean["language"], margins=True))

Saved 974 in-scope rows to data/processed/kaggle_in_scope.csv
Saved 60 out-of-scope rows to data/processed/kaggle_out_of_scope.csv

=== CATEGORY x LANGUAGE COVERAGE FROM KAGGLE ===
language                 English  Hindi  Hinglish  All
category                                              
Drainage & Sewage              0     28        35   63
Municipal Certificates         0    100       120  220
Public Health & Hygiene        0     22         0   22
Roads & Potholes               0    100        95  195
Streetlights                  11     24        13   48
Waste & Sanitation             0     71       110  181
Water Supply                  20     94       131  245
All                           31    439       504  974
